In [ ]:
# This is the notebook file for Multi Head Attention FPGA test

from pynq import Overlay, allocate
import numpy as np

# ---- RTL-defined Matrix Parameters (it should be the same iwth config.svh file) -----
SYSTEM_TOP_WIDTH_INPUT      = 16    # Input width
SYSTEM_TOP_FRAC_WIDTH_INPUT = 8
SYSTEM_TOP_WIDTH_WEIGHT     = 16    # Weight width
SYSTEM_FRAC_WIDTH_WEIGHT    = 8
SYSTEM_TOP_WIDTH_FINAL      = 16    # Output width
SYSTEM_FRAC_WIDTH_FINAL     = 8
I_MATRIX_DIMENSION          = 16    # Input matrix
INNER_MATRIX_DIMENSION      = 10
W_MATRIX_DIMENSION          = 12    # Weight matrix
SYSTEM_NUM_CORES_A          = 2     # Number of cores A
SYSTEM_TOTAL_MODULES        = 2     # Number of cores B
TOP_BLOCK_SIZE              = 2
TOP_CHUNK_SIZE              = 4

# --- FPGA and Design Parameters ---
INPUT_WIDTH_BITS            = SYSTEM_TOP_WIDTH_INPUT * TOP_CHUNK_SIZE * SYSTEM_NUM_CORES_A
IN_DATA_UNIT_BITS           = 16  # uint16, same with SYSTEM_TOP_WIDTH_INPUT
WORDS_PER_INPUT             = INPUT_WIDTH_BITS // IN_DATA_UNIT_BITS

OUTPUT_WIDTH_BITS           = SYSTEM_TOP_WIDTH_FINAL * TOP_CHUNK_SIZE * SYSTEM_NUM_CORES_A * SYSTEM_TOTAL_MODULES * SYSTEM_TOTAL_MODULES 
OUT_DATA_UNIT_BITS          = 16  # uint16, SYSTEM_TOP_WIDTH_FINAL
WORDS_PER_OUTPUT            = OUTPUT_WIDTH_BITS // OUT_DATA_UNIT_BITS

if OUT_DATA_UNIT_BITS == 16:
    out_dtype = np.uint16
elif OUT_DATA_UNIT_BITS == 32:
    out_dtype = np.uint32
elif OUT_DATA_UNIT_BITS == 64:
    out_dtype = np.uint64
else:
    raise ValueError("Unsupported output width")

# --- Derived output size ---
ROWS                        = I_MATRIX_DIMENSION // (TOP_BLOCK_SIZE * SYSTEM_NUM_CORES_A * 2)   # ROW_SIZE_MAT_C_B1
COLS                        = W_MATRIX_DIMENSION // (1 * SYSTEM_TOTAL_MODULES * 2)              # COL_SIZE_MAT_C_B1
TOTAL_OUTPUT_WORDS          = (ROWS * COLS)
OUTPUT_BUFFER_WORDS         = TOTAL_OUTPUT_WORDS * WORDS_PER_OUTPUT

# File paths
INPUT_MEM_FILE              = "i.mem"
OUTPUT_MEM_FILE             = "o.mem"

# --- Load Overlay ---
overlay = Overlay("/home/xilinx/jupyter_notebooks/Matrix_Multiplier/design_1.bit")
print("Overlay loaded.")

In [ ]:
dma_i0  = overlay.axi_dma_0
dma_i1  = overlay.axi_dma_1
dma_o   = overlay.axi_dma_2

In [ ]:
# --- Load HEX .mem File ---
def load_mem_file(filename, word_bits, data_unit_bits):
    with open(filename, "r") as f:
        hex_data = f.read().replace("\n", "").strip()

    word_hex_len = word_bits // 4
    unit_hex_len = data_unit_bits // 4

    chunks = [
        hex_data[i:i+word_hex_len]
        for i in range(0, len(hex_data), word_hex_len)
    ]

    data = []

    for word in chunks:
        word = word.zfill(word_hex_len)

        # Extract from LSB to MSB
        for i in range(0, len(word), unit_hex_len):
            start = len(word) - unit_hex_len - i
            end   = len(word) - i
            data.append(int(word[start:end], 16))

    if data_unit_bits == 16:
        dtype = np.uint16
    elif data_unit_bits == 32:
        dtype = np.uint32
    elif data_unit_bits == 64:
        dtype = np.uint64
    else:
        raise ValueError(f"Unsupported DATA_UNIT_BITS = {data_unit_bits}")

    return np.array(data, dtype=dtype)

In [ ]:
# --- Split even and odd rows ---
def split_even_odd_rows(data, words_per_row):
    rows = data.reshape((-1, words_per_row))
    even = rows[0::2].flatten()
    odd  = rows[1::2].flatten()
    return even, odd

In [ ]:
# --- Load Inputs ---
input_data = load_mem_file(INPUT_MEM_FILE, INPUT_WIDTH_BITS, IN_DATA_UNIT_BITS)

input0_data, input1_data = split_even_odd_rows(
    input_data,
    WORDS_PER_INPUT
)

In [ ]:
# --- Allocate Buffers ---
input0_buffer = allocate(
    shape=input0_data.shape,
    dtype=input0_data.dtype
)

input1_buffer = allocate(
    shape=input1_data.shape,
    dtype=input1_data.dtype
)

np.copyto(input0_buffer, input0_data)
np.copyto(input1_buffer, input1_data)

input0_buffer.flush()
input1_buffer.flush()

output_buffer = allocate(
    shape=(OUTPUT_BUFFER_WORDS,),
    dtype=out_dtype
)

In [ ]:
print(f"Input buffer size: {input0_buffer.nbytes} bytes ({input0_buffer.shape[0]} words)")
print(f"Input buffer size: {input1_buffer.nbytes} bytes ({input1_buffer.shape[0]} words)")
print(f"Output buffer size: {output_buffer.nbytes} bytes ({output_buffer.shape[0]} words)")

In [ ]:
def print_hex_chunks(data_array, bits_per_word, label):
    bits_per_element = data_array.dtype.itemsize * 8
    words_per_line = bits_per_word // bits_per_element
    print(f"\n{label} contents ({bits_per_word}-bit words):")
    reshaped = data_array.reshape((-1, words_per_line))
    for i, line in enumerate(reshaped):
        hex_digits = bits_per_element // 4
        hex_word = ''.join(
            f"{x:0{hex_digits}x}"
            for x in reversed(line)
        )
        print(f"{label} {i}: {hex_word}")

print_hex_chunks(input0_buffer, INPUT_WIDTH_BITS, "Input 0")
print_hex_chunks(input1_buffer, INPUT_WIDTH_BITS, "Input 1")

In [ ]:
# --- Start Transfers ---
dma_o.recvchannel.transfer(output_buffer)
dma_i0.sendchannel.transfer(input0_buffer)
dma_i1.sendchannel.transfer(input1_buffer)

dma_i0.sendchannel.wait()
dma_i1.sendchannel.wait()
dma_o.recvchannel.wait()

In [ ]:
output_buffer.invalidate()
print_hex_chunks(output_buffer, OUTPUT_WIDTH_BITS, "Output")

In [ ]:
# --- Reconstruct Output Data ---
output_chunks = output_buffer.reshape((-1, WORDS_PER_OUTPUT))
hex_digits = output_buffer.dtype.itemsize * 2

output_words = [
    ''.join(
        f"{x:0{hex_digits}x}"
        for x in reversed(chunk)
    )
    for chunk in output_chunks
]

print("Output received (128-bit hex):")
for i, word in enumerate(output_words):
    print(f"Output {i}: {word}")


In [ ]:
with open(OUTPUT_MEM_FILE, "w") as f:
    for word in output_words:
        f.write(word + "\n")

print(f"Saved output to {OUTPUT_MEM_FILE}")

In [ ]:
# Delete buffer to prevent memory leak
del input0_buffer, input1_buffer, output_buffer